# Week 3 Day 4 — Implementing MLP / CNN Cleanly
**Jul 16, 2026**

Every model in this curriculum so far has hardcoded its exact number of layers (`nn.Linear(2, 8)`, `nn.Linear(8, 1)`, done). Want a 3-hidden-layer MLP instead of 1? Rewrite the class. Today: build genuinely reusable, configurable `MLP` and `CNN` classes — and see a real, silent bug that shows up if you build them the naive way.

Scaffold as usual: TODO stubs, hints not formulas, self-check cells.

## Part 1: A tempting but broken approach

Given — watch this fail before building it correctly yourself. The obvious way to hold a variable number of layers is a plain Python list. Here's what that actually does.

In [1]:
import torch
import torch.nn as nn

class BadMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        self.layers = []  # <-- plain python list
        prev_dim = input_dim
        for h in hidden_dims:
            self.layers.append(nn.Linear(prev_dim, h))
            prev_dim = h
        self.out = nn.Linear(prev_dim, output_dim)

    def forward(self, x):
        for layer in self.layers:
            x = torch.relu(layer(x))
        return self.out(x)

model = BadMLP(4, [8, 8], 1)
n_params = sum(p.numel() for p in model.parameters())
expected = (4*8+8) + (8*8+8) + (8*1+1)  # input->8, 8->8, 8->1, each with a bias
print(f"params .parameters() finds: {n_params}   params that actually exist: {expected}")

# does the optimizer even touch the hidden layers?
opt = torch.optim.SGD(model.parameters(), lr=0.1)
w_before = model.layers[0].weight.clone()
x, y = torch.randn(5, 4), torch.randn(5, 1)
loss = ((model(x) - y) ** 2).mean()
loss.backward()
opt.step()
w_after = model.layers[0].weight
print("hidden layer weight changed after optimizer.step()?", not torch.equal(w_before, w_after))

params .parameters() finds: 9   params that actually exist: 121
hidden layer weight changed after optimizer.step()? False


No error anywhere. `model(x)` runs fine, `loss.backward()` runs fine, `optimizer.step()` runs fine — and the hidden layers never train, forever, silently. This is the plain-list version of the `self.x = ...` registration mechanism from Week 1 Day 5 (`nn.Module.__setattr__` only auto-registers submodules assigned directly as attributes) — a plain Python list holding `nn.Linear` objects doesn't trigger that registration, so `.parameters()` never finds what's inside it. `self.out` was assigned directly, so it *is* found — which is exactly why this bug is so dangerous: the model appears to work, produces output, even improves slightly (from `self.out` alone), while most of it does nothing at all.

## Part 2: A correctly reusable `MLP`

TODO: build `MLP(input_dim, hidden_dims, output_dim)` that fixes the bug above — same configurable-depth idea, but every layer actually trains.

The fix: `nn.ModuleList` is a container built specifically so that `nn.Module`'s registration mechanism *does* see what's inside it, unlike a plain list. Build the list of `Linear` layers, wrap it in `nn.ModuleList`, and loop over it in `forward` exactly like you looped over the plain list above.

In [3]:
import torch.nn.functional as F
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        # TODO: build the layers, store them in an nn.ModuleList
        self.layers = nn.ModuleList()
        prev_dim = input_dim
        for h in hidden_dims:
            self.layers.append(nn.Linear(prev_dim, h))
            prev_dim = h
        self.out = nn.Linear(prev_dim, output_dim)

    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        return self.out(x)

# self-check: same architecture as BadMLP above, but correct
model = MLP(4, [8, 8], 1)
n_params = sum(p.numel() for p in model.parameters())
expected = (4*8+8) + (8*8+8) + (8*1+1)
assert n_params == expected, f"expected {expected} params, got {n_params} -- hidden layers not registered?"

opt = torch.optim.SGD(model.parameters(), lr=0.1)
w_before = list(model.parameters())[0].clone()
loss = ((model(x) - y) ** 2).mean()
loss.backward()
opt.step()
w_after = list(model.parameters())[0]
assert not torch.equal(w_before, w_after), "first layer's weight didn't change -- still not registered"

# should also work with a different depth, with no code changes
model2 = MLP(4, [16, 16, 16], 2)
print("3-hidden-layer variant params:", sum(p.numel() for p in model2.parameters()))
print("MLP OK -- correct param count, weights train, and depth is configurable")

3-hidden-layer variant params: 658
MLP OK -- correct param count, weights train, and depth is configurable


## Part 3: A reusable `CNN`

TODO: build `CNN(in_channels, channel_list, n_classes)` — a configurable number of conv blocks, each `Conv2d -> ReLU -> MaxPool2d(2)`, followed by a classification head.

Same `nn.ModuleList` pattern as Part 2. A few new pieces:
- Each `Conv2d` takes `(prev_channels, channels, kernel_size=3, padding=1)` — `padding=1` with a `3x3` kernel keeps the spatial size unchanged by the conv itself (only `MaxPool2d(2)` shrinks it, by half each time).
- After the conv blocks, you need to turn a `(batch, channels, height, width)` tensor into `(batch, channels)` before the final `Linear` head. Don't hardcode the flattened size (`channels * height * width`) — it depends on the input image size *and* how many pooling layers you have, so hardcoding it breaks the moment either changes. `nn.AdaptiveAvgPool2d((1, 1))` collapses the spatial dimensions to exactly `1x1` regardless of what they were, so `.flatten(1)` afterward always gives you a clean `(batch, channels)`, no matter the input resolution.
- The head is a single `nn.Linear(last_channel_count, n_classes)`.

In [4]:
class CNN(nn.Module):
    def __init__(self, in_channels, channel_list, n_classes):
        super().__init__()
        # TODO: build conv blocks in an nn.ModuleList, plus self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.layers = nn.ModuleList()
        prev_channels = in_channels
        for channels in channel_list:
            self.layers.append(nn.Conv2d(prev_channels, channels, kernel_size=3, padding=1))
            prev_channels = channels
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Linear(prev_channels, n_classes)

    def forward(self, x):
        # TODO: run through the blocks, pool, flatten, head
        for layer in self.layers:
            x = F.relu(layer(x))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.head(x)
      

# self-check: works on 28x28 input
cnn = CNN(in_channels=1, channel_list=[8, 16, 32], n_classes=3)
out = cnn(torch.randn(5, 1, 28, 28))
assert out.shape == (5, 3), f"expected (5, 3), got {tuple(out.shape)}"

# self-check: SAME model, DIFFERENT input size -- should still just work
out2 = cnn(torch.randn(5, 1, 32, 32))
assert out2.shape == (5, 3), f"expected (5, 3) on a different input size too, got {tuple(out2.shape)}"

print("CNN OK -- works across input sizes without any hardcoded flatten dimension")

CNN OK -- works across input sizes without any hardcoded flatten dimension


## Try yourself

1. Add an `activation` constructor argument to `MLP` (defaulting to `nn.ReLU`), so `MLP(4, [8,8], 1, activation=nn.Tanh)` swaps the nonlinearity without touching the class body.
2. Add a `kernel_sizes` list to `CNN` (one per block, instead of a fixed `3` everywhere), and figure out what `padding` value keeps spatial size unchanged for a `5x5` kernel (the same logic that made `padding=1` work for `3x3`).
3. Compare parameter counts: build an `MLP` and a `CNN` that both take a flattened `28x28` image as input and produce `10` output classes (`MLP(784, [128], 10)` vs. a small `CNN`). Which has fewer parameters, and why does convolution's weight-sharing make that true in general for image-shaped data?
4. Add a `dropout` argument to `MLP` that inserts `nn.Dropout(p)` after each hidden activation, reusing what you built in Day 2 — configurable depth *and* configurable regularization, without hardcoding either.